# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. All exploration refers to dataset components using their `@id` fields as required for reproducibility and clarity.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- Croissant schema URL: https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print out key metadata summary
print(f"{metadata.name}: {metadata.description}\n\nPublished: {metadata.datePublished}\nIdentifier: {metadata.identifier}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the schema. All references will be by their `@id` fields.

> Note: To ensure reproducibility, the notebook dynamically lists record sets and their fields using their `@id`s. You may need to browse through the metadata to identify which record sets contain tabular or regression data for analysis.

In [ ]:
# Examine the record sets in the dataset using their '@id'.
from collections import defaultdict

recordsets = list(dataset.record_sets())
print(f"Found {len(recordsets)} record set(s) in the dataset.\n")

recordsets_info = defaultdict(list)
for rs in recordsets:
    print(f"Record set @id: {rs['@id']}")
    fields = rs['fields'] if 'fields' in rs else []
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        # field can be a dict with '@id', 'name', 'dataType', etc.
        fname = field.get('name', field.get('@id', ''))
        ftype = field.get('dataType', '')
        print(f"  Field @id: {field.get('@id', '')}, Name: {fname}, Type: {ftype}")
        recordsets_info[rs['@id']].append(field.get('@id', ''))

# List all found record set @ids
record_set_ids = [rs['@id'] for rs in recordsets]
print(f"\nRecord set @ids: {record_set_ids}")

## 3. Data Extraction
Load data from the record set(s) using their `@id` fields into a pandas DataFrame for inspection and further analysis.

> By default, we extract all available record sets. Modify to analyze a specific record set by its `@id`.

In [ ]:
# Extract data from each record set

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records from record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print("  No records found in this record set.")
    except Exception as e:
        print(f"  Error loading records for {record_set_id}: {e}")

# Select a target record set id for further steps (e.g., first with data)
main_rs_id = None
for rs_id, df in dataframes.items():
    if not df.empty:
        main_rs_id = rs_id
        break
if main_rs_id:
    print(f"Using record set '@id': {main_rs_id} for further analysis.")
    print("Available columns:", dataframes[main_rs_id].columns.tolist())
else:
    print("No record set with data available.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering records based on numeric criteria, normalizing fields, and grouping data. Record set and field references use `@id`s.

> Adjust the `numeric_field_id` and `group_field_id` variables to reference appropriate field `@id`s shown in prior outputs.

In [ ]:
# EDA: Only proceed if a numeric field exists in main_rs_id DataFrame
import numpy as np

if main_rs_id:
    # Attempt to automatically select a numeric field based on type or column names
    df = dataframes[main_rs_id]
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Selected numeric field for analysis: {numeric_field_id}")
    else:
        print("No numeric fields found for EDA. Please specify a field manually.")
        numeric_field_id = None

    # Example threshold to filter data
    threshold = 10

    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with '{numeric_field_id}' > {threshold} (showing up to 5 rows):")
        display(filtered_df.head())

        # Normalize numeric field
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std

        print(f"\nNormalized '{numeric_field_id}' (showing up to 5 rows):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a candidate categorical field (if any)
        group_candidates = [col for col in df.columns if col != numeric_field_id and (df[col].dtype == 'object' or df[col].dtype.name == 'category')]
        group_field_id = group_candidates[0] if group_candidates else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}' (showing up to 5 rows):")
            display(grouped_df.head())
        else:
            print("No categorical columns found for grouping.")
else:
    print("No main record set loaded; skipping EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Adjust field references as needed. All visualizations will refer to fields using their `@id` where possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distributions if available
if main_rs_id and numeric_field_id:
    plt.figure(figsize=(8,5))
    sns.histplot(dataframes[main_rs_id][numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=dataframes[main_rs_id][group_field_id], y=dataframes[main_rs_id][numeric_field_id])
        plt.xticks(rotation=45)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to programmatically explore a FAIR^2 dataset described in Croissant format with `mlcroissant`, referencing all dataset elements by their `@id`. Fields and record sets were dynamically discovered for robust, schema-driven analysis; additional domain-driven statistics may be performed once domain-specific field types are understood. 

**Next steps:** Perform domain-informed analyses (e.g., regression summarization, categorical variable inspection, detailed missing data analysis) and consult dataset documentation for additional guidance on variable meanings.